In [1]:
import os

os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGCHAIN_TRACING_V2"] = "false"

# Optional stronger disable for this notebook session
os.environ.pop("LANGSMITH_API_KEY", None)
os.environ.pop("LANGCHAIN_API_KEY", None)

print("LANGSMITH_TRACING:", os.environ.get("LANGSMITH_TRACING"))
print("LANGCHAIN_TRACING_V2:", os.environ.get("LANGCHAIN_TRACING_V2"))
print("LANGSMITH_API_KEY exists:", bool(os.environ.get("LANGSMITH_API_KEY")))
print("LANGCHAIN_API_KEY exists:", bool(os.environ.get("LANGCHAIN_API_KEY")))

LANGSMITH_TRACING: false
LANGCHAIN_TRACING_V2: false
LANGSMITH_API_KEY exists: False
LANGCHAIN_API_KEY exists: False


In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

project_root = Path.cwd()
if project_root.name == "notebook":
    project_root = project_root.parent

env_path = project_root / ".env"
load_dotenv(env_path, override=True)

print("ENV path:", env_path)
print("ENV exists:", env_path.exists())
print("OPENAI_API_KEY exists:", bool(os.getenv("OPENAI_API_KEY")))

# LangSmith is optional for now because API access is returning 403.
print("LANGSMITH_API_KEY exists:", bool(os.getenv("LANGSMITH_API_KEY")))
print("LANGSMITH_ENDPOINT:", os.getenv("LANGSMITH_ENDPOINT"))
print("LANGSMITH_WORKSPACE_ID:", os.getenv("LANGSMITH_WORKSPACE_ID"))

ENV path: c:\faizul-personal\chatbot-evaluation-langsmith-krishnaik-YT\chatbot-evaluator\.env
ENV exists: True
OPENAI_API_KEY exists: True
LANGSMITH_API_KEY exists: True
LANGSMITH_ENDPOINT: https://aws.api.smith.langchain.com
LANGSMITH_WORKSPACE_ID: None


In [3]:
from dotenv import load_dotenv
from pathlib import Path
import os

project_root = Path.cwd()

if project_root.name.lower() == "notebook":
    project_root = project_root.parent

env_path = project_root / ".env"
load_dotenv(env_path, override=True)

print("ENV path:", env_path)
print("ENV exists:", env_path.exists())
print("LANGSMITH_ENDPOINT:", os.getenv("LANGSMITH_ENDPOINT"))
print("LANGSMITH_API_KEY exists:", bool(os.getenv("LANGSMITH_API_KEY")))
print("LANGSMITH_PROJECT:", os.getenv("LANGSMITH_PROJECT"))
print("LANGSMITH_TRACING:", os.getenv("LANGSMITH_TRACING"))

ENV path: c:\faizul-personal\chatbot-evaluation-langsmith-krishnaik-YT\chatbot-evaluator\.env
ENV exists: True
LANGSMITH_ENDPOINT: https://aws.api.smith.langchain.com
LANGSMITH_API_KEY exists: True
LANGSMITH_PROJECT: chatbot-evaluator
LANGSMITH_TRACING: true


In [4]:
from langsmith import Client
import os

client = Client(
    api_key=os.getenv("LANGSMITH_API_KEY"),
    api_url=os.getenv("LANGSMITH_ENDPOINT"),
)

print("Client API URL:", client.api_url)

projects = list(client.list_projects(limit=3))
print("Projects:")
for project in projects:
    print("-", project.name)

Client API URL: https://aws.api.smith.langchain.com
Projects:


In [5]:
datasets = list(client.list_datasets(limit=5))

print("Datasets:")
for dataset in datasets:
    print("-", dataset.name)

Datasets:


In [6]:
dataset_name = "Chatbots Evaluation"

examples = [
    {
        "inputs": {"question": "What is LangChain?"},
        "outputs": {"answer": "A framework for building LLM applications"},
    },
    {
        "inputs": {"question": "What is LangSmith?"},
        "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
    },
    {
        "inputs": {"question": "What is OpenAI?"},
        "outputs": {"answer": "A company that creates Large Language Models"},
    },
    {
        "inputs": {"question": "What is Google?"},
        "outputs": {"answer": "A technology company known for search"},
    },
    {
        "inputs": {"question": "What is Mistral?"},
        "outputs": {"answer": "A company that creates Large Language Models"},
    },
    {
        "inputs": {"question": "What is Cefalo?"},
        "outputs": {"answer": "A software company that provides development teams and services"},
    },
]

print(f"Local dataset ready: {dataset_name}")
print(f"Total examples: {len(examples)}")

Local dataset ready: Chatbots Evaluation
Total examples: 6


In [7]:
dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Simple chatbot evaluation dataset with question/reference answer pairs.",
)

inputs = [example["inputs"] for example in examples]
outputs = [example["outputs"] for example in examples]

client.create_examples(
    dataset_id=dataset.id,
    inputs=inputs,
    outputs=outputs,
)

print(f"Created LangSmith dataset: {dataset.name}")
print(f"Pushed {len(examples)} examples")

Created LangSmith dataset: Chatbots Evaluation
Pushed 6 examples


In [8]:
datasets = list(client.list_datasets(limit=5))

print("Datasets:")
for dataset in datasets:
    print("-", dataset.name)

Datasets:
- Chatbots Evaluation


In [9]:
langsmith_examples = list(client.list_examples(dataset_id=dataset.id))

print("Total examples:", len(langsmith_examples))

for ex in langsmith_examples[:3]:
    print("INPUT:", ex.inputs)
    print("OUTPUT:", ex.outputs)
    print("---")

Total examples: 6
INPUT: {'question': 'What is Cefalo?'}
OUTPUT: {'answer': 'A software company that provides development teams and services'}
---
INPUT: {'question': 'What is OpenAI?'}
OUTPUT: {'answer': 'A company that creates Large Language Models'}
---
INPUT: {'question': 'What is Mistral?'}
OUTPUT: {'answer': 'A company that creates Large Language Models'}
---


In [10]:
from openai import OpenAI
import os

openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def my_app(question: str, model: str = "gpt-4o-mini") -> str:
    response = openai_client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": "Answer the user's question clearly and concisely."
            },
            {
                "role": "user",
                "content": question
            }
        ],
    )
    return response.choices[0].message.content

In [11]:
def target(inputs: dict) -> dict:
    return {
        "response": my_app(inputs["question"])
    }

In [12]:
def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    reference = reference_outputs["answer"].lower()
    response = outputs["response"].lower()

    reference_words = set(reference.split())
    response_words = set(response.split())

    overlap = reference_words.intersection(response_words)

    return len(overlap) >= max(1, len(reference_words) // 2)


def concision(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    return len(outputs["response"].split()) <= 30

In [13]:
experiment_results = client.evaluate(
    target,
    data="Chatbots Evaluation",
    evaluators=[correctness, concision],
    experiment_prefix="openai-chatbot-eval",
)

c:\faizul-personal\chatbot-evaluation-langsmith-krishnaik-YT\chatbot-evaluator\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'openai-chatbot-eval-aec404fd' at:
https://smith.langchain.com/o/cfd96714-2d44-452b-b186-e66160e512d3/datasets/2f72cf65-4618-496f-abdc-39e19a390c15/compare?selectedSessions=5bfd17e8-c00b-4829-bd96-a1603f31b84e




6it [00:16,  2.73s/it]


In [7]:
dataset_name = "Chatbots Evaluation"

examples = [
    {
        "inputs": {"question": "What is LangChain?"},
        "outputs": {"answer": "A framework for building LLM applications"},
    },
    {
        "inputs": {"question": "What is LangSmith?"},
        "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
    },
    {
        "inputs": {"question": "What is OpenAI?"},
        "outputs": {"answer": "A company that creates Large Language Models"},
    },
    {
        "inputs": {"question": "What is Google?"},
        "outputs": {"answer": "A technology company known for search"},
    },
    {
        "inputs": {"question": "What is Mistral?"},
        "outputs": {"answer": "A company that creates Large Language Models"},
    },
    {
        "inputs": {"question": "What is Cefalo?"},
        "outputs": {"answer": "A software company that provides development teams and services"},
    },
]

print(f"Local dataset ready: {dataset_name}")
print(f"Total examples: {len(examples)}")

Local dataset ready: Chatbots Evaluation
Total examples: 6


In [8]:
from openai import OpenAI

openai_client = OpenAI()

In [9]:
default_instructions = "Respond to the user's question in a short, concise manner."

def my_app(
    question: str,
    model: str = "gpt-4o-mini",
    instructions: str = default_instructions,
) -> str:
    response = openai_client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": question},
        ],
    )

    return response.choices[0].message.content

In [10]:
my_app("What is LangChain?")

'LangChain is a framework designed for building applications that utilize language models. It provides tools and components to facilitate the integration of language models with various data sources, APIs, and workflows, enabling developers to create more complex and interactive applications.'

In [12]:
my_app("What is Cefalo?")

'Cefalo is a technology company that specializes in software development, IT consulting, and digital transformation services. They focus on delivering innovative solutions to enhance business processes and improve efficiency.'

In [13]:
my_app("What is cricket?")

'Cricket is a bat-and-ball sport played between two teams of eleven players each. It involves batting, bowling, and fielding, with the objective of scoring runs and dismissing the opposing team. The game is played on a circular or oval field, with a 22-yard long pitch at the center.'

In [15]:
###define evaluator 

eval_instructions = "You are an expert evaluator. Grade whether the predicted answer is semantically correct compared to the reference answer."

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    predicted_answer = outputs.get("response") or outputs.get("answer")

    user_content = f"""
Question:
{inputs['question']}

Reference answer:
{reference_outputs['answer']}

Predicted answer:
{predicted_answer}

Is the predicted answer semantically correct compared to the reference answer?

Respond with only one word:
CORRECT or INCORRECT
"""

    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {"role": "system", "content": eval_instructions},
            {"role": "user", "content": user_content},
        ],
    )

    verdict = response.choices[0].message.content.strip().upper()
    return verdict == "CORRECT"


def concision(inputs: dict, outputs: dict, reference_outputs: dict = None) -> bool:
    predicted_answer = outputs.get("response") or outputs.get("answer")
    return len(predicted_answer.split()) <= 30

In [18]:
### Run local evaluation
import pandas as pd

local_results = []

for example in examples:
    inputs = example["inputs"]
    reference_outputs = example["outputs"]

    model_response = my_app(inputs["question"])

    outputs = {
        "response": model_response
    }

    result = {
        "question": inputs["question"],
        "reference_answer": reference_outputs["answer"],
        "model_response": model_response,
        "correctness": correctness(inputs, outputs, reference_outputs),
        "concision": concision(inputs, outputs, reference_outputs),
    }

    local_results.append(result)

results_df = pd.DataFrame(local_results)
results_df

,question,reference_answer,model_response,correctness,concision
0,What is LangChain?,A framework for building LLM applications,LangChain is a framework designed for building...,True,False
1,What is LangSmith?,A platform for observing and evaluating LLM ap...,LangSmith is a platform designed for developer...,True,False
2,What is OpenAI?,A company that creates Large Language Models,OpenAI is an artificial intelligence research ...,True,False
3,What is Google?,A technology company known for search,Google is a multinational technology company s...,True,False
4,What is Mistral?,A company that creates Large Language Models,Mistral is a company that develops advanced AI...,True,False
5,What is Cefalo?,A software company that provides development t...,Cefalo is a technology company that specialize...,True,True


In [19]:
def ls_target(inputs: dict) -> dict:
    return {"response": my_app(inputs["question"])}

In [22]:
import pandas as pd

def run_local_evaluation(model_name: str, experiment_name: str):
    local_results = []

    for example in examples:
        inputs = example["inputs"]
        reference_outputs = example["outputs"]

        model_response = my_app(
            question=inputs["question"],
            model=model_name,
        )

        outputs = {
            "response": model_response
        }

        result = {
            "experiment": experiment_name,
            "model": model_name,
            "question": inputs["question"],
            "reference_answer": reference_outputs["answer"],
            "model_response": model_response,
            "correctness": correctness(inputs, outputs, reference_outputs),
            "concision": concision(inputs, outputs, reference_outputs),
        }

        local_results.append(result)

    return pd.DataFrame(local_results)

In [23]:
results_gpt_4o_mini = run_local_evaluation(
    model_name="gpt-4o-mini",
    experiment_name="openai-4o-mini-chatbot",
)

results_gpt_4o_mini

,experiment,model,question,reference_answer,model_response,correctness,concision
0,openai-4o-mini-chatbot,gpt-4o-mini,What is LangChain?,A framework for building LLM applications,LangChain is a framework designed for building...,True,False
1,openai-4o-mini-chatbot,gpt-4o-mini,What is LangSmith?,A platform for observing and evaluating LLM ap...,LangSmith is a platform designed for developer...,True,False
2,openai-4o-mini-chatbot,gpt-4o-mini,What is OpenAI?,A company that creates Large Language Models,OpenAI is an artificial intelligence research ...,True,False
3,openai-4o-mini-chatbot,gpt-4o-mini,What is Google?,A technology company known for search,Google is a multinational technology company s...,True,False
4,openai-4o-mini-chatbot,gpt-4o-mini,What is Mistral?,A company that creates Large Language Models,Mistral is a company that develops advanced AI...,True,False
5,openai-4o-mini-chatbot,gpt-4o-mini,What is Cefalo?,A software company that provides development t...,Cefalo is a company that specializes in softwa...,True,False


In [24]:
results_gpt_4_turbo = run_local_evaluation(
    model_name="gpt-4-turbo",
    experiment_name="openai-4-turbo-chatbot",
)

results_gpt_4_turbo

,experiment,model,question,reference_answer,model_response,correctness,concision
0,openai-4-turbo-chatbot,gpt-4-turbo,What is LangChain?,A framework for building LLM applications,LangChain is a software library designed to fa...,True,False
1,openai-4-turbo-chatbot,gpt-4-turbo,What is LangSmith?,A platform for observing and evaluating LLM ap...,LangSmith is an AI-powered writing assistant d...,False,False
2,openai-4-turbo-chatbot,gpt-4-turbo,What is OpenAI?,A company that creates Large Language Models,OpenAI is an artificial intelligence research ...,True,False
3,openai-4-turbo-chatbot,gpt-4-turbo,What is Google?,A technology company known for search,Google is a multinational technology company t...,True,False
4,openai-4-turbo-chatbot,gpt-4-turbo,What is Mistral?,A company that creates Large Language Models,"Mistral is a strong, cold, and dry regional wi...",False,False
5,openai-4-turbo-chatbot,gpt-4-turbo,What is Cefalo?,A software company that provides development t...,Cefalo is a technology company based in Norway...,True,False


In [25]:
results_gpt_4_1_mini = run_local_evaluation(
    model_name="gpt-4.1-mini",
    experiment_name="openai-4-1-mini-chatbot",
)

results_gpt_4_1_mini

,experiment,model,question,reference_answer,model_response,correctness,concision
0,openai-4-1-mini-chatbot,gpt-4.1-mini,What is LangChain?,A framework for building LLM applications,LangChain is a framework designed to help deve...,True,True
1,openai-4-1-mini-chatbot,gpt-4.1-mini,What is LangSmith?,A platform for observing and evaluating LLM ap...,LangSmith is a platform by LangChain designed ...,True,True
2,openai-4-1-mini-chatbot,gpt-4.1-mini,What is OpenAI?,A company that creates Large Language Models,OpenAI is an artificial intelligence research ...,True,True
3,openai-4-1-mini-chatbot,gpt-4.1-mini,What is Google?,A technology company known for search,Google is a multinational technology company s...,True,True
4,openai-4-1-mini-chatbot,gpt-4.1-mini,What is Mistral?,A company that creates Large Language Models,Mistral is an open-source large language model...,True,True
5,openai-4-1-mini-chatbot,gpt-4.1-mini,What is Cefalo?,A software company that provides development t...,Cefalo is a brand name for the antibiotic drug...,False,True


In [29]:
comparison_df = pd.concat(
    [results_gpt_4o_mini, results_gpt_4_turbo, results_gpt_4_1_mini],
    ignore_index=True,
)

comparison_df

,experiment,model,question,reference_answer,model_response,correctness,concision
0,openai-4o-mini-chatbot,gpt-4o-mini,What is LangChain?,A framework for building LLM applications,LangChain is a framework designed for building...,True,False
1,openai-4o-mini-chatbot,gpt-4o-mini,What is LangSmith?,A platform for observing and evaluating LLM ap...,LangSmith is a platform designed for developer...,True,False
2,openai-4o-mini-chatbot,gpt-4o-mini,What is OpenAI?,A company that creates Large Language Models,OpenAI is an artificial intelligence research ...,True,False
3,openai-4o-mini-chatbot,gpt-4o-mini,What is Google?,A technology company known for search,Google is a multinational technology company s...,True,False
4,openai-4o-mini-chatbot,gpt-4o-mini,What is Mistral?,A company that creates Large Language Models,Mistral is a company that develops advanced AI...,True,False
5,openai-4o-mini-chatbot,gpt-4o-mini,What is Cefalo?,A software company that provides development t...,Cefalo is a company that specializes in softwa...,True,False
6,openai-4-turbo-chatbot,gpt-4-turbo,What is LangChain?,A framework for building LLM applications,LangChain is a software library designed to fa...,True,False
7,openai-4-turbo-chatbot,gpt-4-turbo,What is LangSmith?,A platform for observing and evaluating LLM ap...,LangSmith is an AI-powered writing assistant d...,False,False
8,openai-4-turbo-chatbot,gpt-4-turbo,What is OpenAI?,A company that creates Large Language Models,OpenAI is an artificial intelligence research ...,True,False
9,openai-4-turbo-chatbot,gpt-4-turbo,What is Google?,A technology company known for search,Google is a multinational technology company t...,True,False


In [30]:
comparison_df = pd.concat(
    [
        results_gpt_4o_mini,
        results_gpt_4_turbo,
        results_gpt_4_1_mini,
    ],
    ignore_index=True,
)

comparison_df

,experiment,model,question,reference_answer,model_response,correctness,concision
0,openai-4o-mini-chatbot,gpt-4o-mini,What is LangChain?,A framework for building LLM applications,LangChain is a framework designed for building...,True,False
1,openai-4o-mini-chatbot,gpt-4o-mini,What is LangSmith?,A platform for observing and evaluating LLM ap...,LangSmith is a platform designed for developer...,True,False
2,openai-4o-mini-chatbot,gpt-4o-mini,What is OpenAI?,A company that creates Large Language Models,OpenAI is an artificial intelligence research ...,True,False
3,openai-4o-mini-chatbot,gpt-4o-mini,What is Google?,A technology company known for search,Google is a multinational technology company s...,True,False
4,openai-4o-mini-chatbot,gpt-4o-mini,What is Mistral?,A company that creates Large Language Models,Mistral is a company that develops advanced AI...,True,False
5,openai-4o-mini-chatbot,gpt-4o-mini,What is Cefalo?,A software company that provides development t...,Cefalo is a company that specializes in softwa...,True,False
6,openai-4-turbo-chatbot,gpt-4-turbo,What is LangChain?,A framework for building LLM applications,LangChain is a software library designed to fa...,True,False
7,openai-4-turbo-chatbot,gpt-4-turbo,What is LangSmith?,A platform for observing and evaluating LLM ap...,LangSmith is an AI-powered writing assistant d...,False,False
8,openai-4-turbo-chatbot,gpt-4-turbo,What is OpenAI?,A company that creates Large Language Models,OpenAI is an artificial intelligence research ...,True,False
9,openai-4-turbo-chatbot,gpt-4-turbo,What is Google?,A technology company known for search,Google is a multinational technology company t...,True,False


In [31]:
summary_df = comparison_df.groupby(["experiment", "model"]).agg(
    total_examples=("question", "count"),
    correctness_score=("correctness", "mean"),
    concision_score=("concision", "mean"),
).reset_index()

summary_df

,experiment,model,total_examples,correctness_score,concision_score
0,openai-4-1-mini-chatbot,gpt-4.1-mini,6,0.833333,1.0
1,openai-4-turbo-chatbot,gpt-4-turbo,6,0.666667,0.0
2,openai-4o-mini-chatbot,gpt-4o-mini,6,1.000000,0.0


In [32]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

docs = []

for url in urls:
    print(f"Loading: {url}")
    loaded_docs = WebBaseLoader(url).load()
    docs.extend(loaded_docs)

print(f"Loaded documents: {len(docs)}")

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250,
    chunk_overlap=0,
)

doc_splits = text_splitter.split_documents(docs)

print(f"Total chunks: {len(doc_splits)}")

vectorstore = InMemoryVectorStore.from_documents(
    documents=doc_splits,
    embedding=OpenAIEmbeddings(),
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 6})

print("Retriever is ready.")

c:\faizul-personal\chatbot-evaluation-langsmith-krishnaik-YT\chatbot-evaluator\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


Loading: https://lilianweng.github.io/posts/2023-06-23-agent/
Loading: https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/
Loading: https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/
Loaded documents: 3
Total chunks: 187
Retriever is ready.


In [37]:
import os

os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGCHAIN_TRACING_V2"] = "false"

print("LANGSMITH_TRACING:", os.environ.get("LANGSMITH_TRACING"))
print("LANGCHAIN_TRACING_V2:", os.environ.get("LANGCHAIN_TRACING_V2"))

LANGSMITH_TRACING: false
LANGCHAIN_TRACING_V2: false


In [4]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

docs = []

for url in urls:
    print(f"Loading: {url}")
    loaded_docs = WebBaseLoader(url).load()
    docs.extend(loaded_docs)

print(f"Loaded documents: {len(docs)}")

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250,
    chunk_overlap=0,
)

doc_splits = text_splitter.split_documents(docs)

print(f"Total chunks: {len(doc_splits)}")

vectorstore = InMemoryVectorStore.from_documents(
    documents=doc_splits,
    embedding=OpenAIEmbeddings(),
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 6})

print("Retriever is ready.")

c:\faizul-personal\chatbot-evaluation-langsmith-krishnaik-YT\chatbot-evaluator\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


Loading: https://lilianweng.github.io/posts/2023-06-23-agent/
Loading: https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/
Loading: https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/
Loaded documents: 3
Total chunks: 187
Retriever is ready.


In [5]:
retrieved_docs = retriever.invoke("what is agents")

print(f"Retrieved docs: {len(retrieved_docs)}")
print(retrieved_docs[0].page_content[:700])

Retrieved docs: 6
They also discussed the risks, especially with illicit drugs and bioweapons. They developed a test set containing a list of known chemical weapon agents and asked the agent to synthesize them. 4 out of 11 requests (36%) were accepted to obtain a synthesis solution and the agent attempted to consult documentation to execute the procedure. 7 out of 11 were rejected and among these 7 rejected cases, 5 happened after a Web search while 2 were rejected based on prompt only.
Generative Agents Simulation#
Generative Agents (Park, et al. 2023) is super fun experiment where 25 virtual characters, each controlled by a LLM-powered agent, are living and interacting in a sandbox environment, inspired by 


In [6]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)

def rag_bot(question: str) -> dict:
    docs = retriever.invoke(question)
    docs_string = "\n\n".join(doc.page_content for doc in docs)

    instructions = f"""
You are a helpful assistant who is good at analyzing source information and answering questions.

Use the following source documents to answer the user's question.
If you don't know the answer, say that you don't know.
Use three sentences maximum and keep the answer concise.

Documents:
{docs_string}
"""

    ai_msg = llm.invoke(
        [
            {"role": "system", "content": instructions},
            {"role": "user", "content": question},
        ]
    )

    return {
        "answer": ai_msg.content,
        "documents": docs,
    }

In [7]:
test_result = rag_bot("How does the ReAct agent use self-reflection?")

print("Answer:")
print(test_result["answer"])

print("\nRetrieved documents:")
print(len(test_result["documents"]))

Answer:
The ReAct agent uses self-reflection by integrating reasoning and acting, allowing it to refine its decision-making through iterative processes. It incorporates a "Thought" step in its prompt template, enabling the agent to evaluate its actions and observations, which helps improve future performance. This self-reflection mechanism is crucial for enhancing the agent's reasoning skills in both knowledge-intensive and decision-making tasks.

Retrieved documents:
6


In [8]:
rag_dataset_name = "RAG Test Evaluation"

rag_examples = [
    {
        "inputs": {
            "question": "How does the ReAct agent use self-reflection?"
        },
        "outputs": {
            "answer": "ReAct integrates reasoning and acting, performing actions such as using tools, then observing and reasoning about the tool outputs."
        },
    },
    {
        "inputs": {
            "question": "What are the types of biases that can arise with few-shot prompting?"
        },
        "outputs": {
            "answer": "The biases that can arise with few-shot prompting include majority label bias, recency bias, and common token bias."
        },
    },
    {
        "inputs": {
            "question": "What are five types of adversarial attacks?"
        },
        "outputs": {
            "answer": "Five types of adversarial attacks include token manipulation, gradient-based attacks, jailbreak prompting, human red-teaming, and model red-teaming."
        },
    },
]

print(f"Local RAG dataset ready: {rag_dataset_name}")
print(f"Total examples: {len(rag_examples)}")

Local RAG dataset ready: RAG Test Evaluation
Total examples: 3


In [9]:
from typing_extensions import Annotated, TypedDict
from langchain_openai import ChatOpenAI


class CorrectnessGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    correct: Annotated[bool, ..., "True if the answer is correct, False otherwise."]


class RelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "True if the answer addresses the question, False otherwise."]


class GroundedGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    grounded: Annotated[bool, ..., "True if the answer is grounded in the documents, False otherwise."]


class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "True if the retrieved documents are relevant to the question, False otherwise."]


grader_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)

correctness_llm = grader_llm.with_structured_output(
    CorrectnessGrade,
    method="json_schema",
    strict=True,
)

relevance_llm = grader_llm.with_structured_output(
    RelevanceGrade,
    method="json_schema",
    strict=True,
)

grounded_llm = grader_llm.with_structured_output(
    GroundedGrade,
    method="json_schema",
    strict=True,
)

retrieval_relevance_llm = grader_llm.with_structured_output(
    RetrievalRelevanceGrade,
    method="json_schema",
    strict=True,
)

In [10]:
correctness_instructions = """
You are a teacher grading a quiz.

You will be given a QUESTION, the GROUND TRUTH ANSWER, and the STUDENT ANSWER.

Grade the student answer based only on factual accuracy relative to the ground truth answer.

Return correct=True if the answer is accurate.
Return correct=False if the answer is inaccurate.
Explain your reasoning briefly.
"""

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    answers = f"""
QUESTION:
{inputs['question']}

GROUND TRUTH ANSWER:
{reference_outputs['answer']}

STUDENT ANSWER:
{outputs['answer']}
"""

    grade = correctness_llm.invoke(
        [
            {"role": "system", "content": correctness_instructions},
            {"role": "user", "content": answers},
        ]
    )

    return grade["correct"]


relevance_instructions = """
You are a teacher grading a quiz.

You will be given a QUESTION and a STUDENT ANSWER.

Return relevant=True if the answer addresses and helps answer the question.
Return relevant=False if the answer is off-topic or unhelpful.
Explain your reasoning briefly.
"""

def relevance(inputs: dict, outputs: dict) -> bool:
    answer = f"""
QUESTION:
{inputs['question']}

STUDENT ANSWER:
{outputs['answer']}
"""

    grade = relevance_llm.invoke(
        [
            {"role": "system", "content": relevance_instructions},
            {"role": "user", "content": answer},
        ]
    )

    return grade["relevant"]


grounded_instructions = """
You are a teacher grading a RAG answer.

You will be given FACTS from retrieved documents and a STUDENT ANSWER.

Return grounded=True if the answer is supported by the FACTS.
Return grounded=False if the answer contains unsupported claims.
Explain your reasoning briefly.
"""

def groundedness(inputs: dict, outputs: dict) -> bool:
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])

    answer = f"""
FACTS:
{doc_string}

STUDENT ANSWER:
{outputs['answer']}
"""

    grade = grounded_llm.invoke(
        [
            {"role": "system", "content": grounded_instructions},
            {"role": "user", "content": answer},
        ]
    )

    return grade["grounded"]


retrieval_relevance_instructions = """
You are a teacher grading document retrieval quality.

You will be given a QUESTION and FACTS retrieved by a RAG system.

Return relevant=True if the retrieved documents contain information related to the QUESTION.
Return relevant=False if the retrieved documents are completely unrelated.
Explain your reasoning briefly.
"""

def retrieval_relevance(inputs: dict, outputs: dict) -> bool:
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])

    answer = f"""
QUESTION:
{inputs['question']}

FACTS:
{doc_string}
"""

    grade = retrieval_relevance_llm.invoke(
        [
            {"role": "system", "content": retrieval_relevance_instructions},
            {"role": "user", "content": answer},
        ]
    )

    return grade["relevant"]

In [11]:
import pandas as pd

def target(inputs: dict) -> dict:
    return rag_bot(inputs["question"])


rag_local_results = []

for example in rag_examples:
    inputs = example["inputs"]
    reference_outputs = example["outputs"]

    outputs = target(inputs)

    result = {
        "question": inputs["question"],
        "reference_answer": reference_outputs["answer"],
        "rag_answer": outputs["answer"],
        "correctness": correctness(inputs, outputs, reference_outputs),
        "groundedness": groundedness(inputs, outputs),
        "relevance": relevance(inputs, outputs),
        "retrieval_relevance": retrieval_relevance(inputs, outputs),
        "retrieved_docs_count": len(outputs["documents"]),
    }

    rag_local_results.append(result)

rag_results_df = pd.DataFrame(rag_local_results)
rag_results_df

,question,reference_answer,rag_answer,correctness,groundedness,relevance,retrieval_relevance,retrieved_docs_count
0,How does the ReAct agent use self-reflection?,"ReAct integrates reasoning and acting, perform...",The ReAct agent uses self-reflection by integr...,True,True,True,True,6
1,What are the types of biases that can arise wi...,The biases that can arise with few-shot prompt...,The types of biases that can arise with few-sh...,True,True,True,True,6
2,What are five types of adversarial attacks?,Five types of adversarial attacks include toke...,The five types of adversarial attacks are toke...,True,True,True,True,6


In [12]:
total = len(rag_results_df)

print(f"Total RAG examples: {total}")
print(f"Correctness score: {rag_results_df['correctness'].mean():.2%}")
print(f"Groundedness score: {rag_results_df['groundedness'].mean():.2%}")
print(f"Answer relevance score: {rag_results_df['relevance'].mean():.2%}")
print(f"Retrieval relevance score: {rag_results_df['retrieval_relevance'].mean():.2%}")

Total RAG examples: 3
Correctness score: 100.00%
Groundedness score: 100.00%
Answer relevance score: 100.00%
Retrieval relevance score: 100.00%


In [13]:
def rag_bot(question: str, model_name: str = "gpt-4o-mini") -> dict:
    docs = retriever.invoke(question)
    docs_string = "\n\n".join(doc.page_content for doc in docs)

    llm = ChatOpenAI(
        model=model_name,
        temperature=0,
    )

    instructions = f"""
You are a helpful assistant who is good at analyzing source information and answering questions.

Use the following source documents to answer the user's question.
If you don't know the answer, say that you don't know.
Use three sentences maximum and keep the answer concise.

Documents:
{docs_string}
"""

    ai_msg = llm.invoke(
        [
            {"role": "system", "content": instructions},
            {"role": "user", "content": question},
        ]
    )

    return {
        "answer": ai_msg.content,
        "documents": docs,
        "model": model_name,
    }

In [14]:
def run_rag_evaluation(model_name: str):
    rag_local_results = []

    for example in rag_examples:
        inputs = example["inputs"]
        reference_outputs = example["outputs"]

        outputs = rag_bot(
            question=inputs["question"],
            model_name=model_name,
        )

        result = {
            "model": model_name,
            "question": inputs["question"],
            "reference_answer": reference_outputs["answer"],
            "rag_answer": outputs["answer"],
            "correctness": correctness(inputs, outputs, reference_outputs),
            "groundedness": groundedness(inputs, outputs),
            "relevance": relevance(inputs, outputs),
            "retrieval_relevance": retrieval_relevance(inputs, outputs),
            "retrieved_docs_count": len(outputs["documents"]),
        }

        rag_local_results.append(result)

    return pd.DataFrame(rag_local_results)

In [15]:
rag_results_gpt_4o_mini = run_rag_evaluation("gpt-4o-mini")
rag_results_gpt_4_1_mini = run_rag_evaluation("gpt-4.1-mini")

In [16]:
rag_comparison_df = pd.concat(
    [
        rag_results_gpt_4o_mini,
        rag_results_gpt_4_1_mini,
    ],
    ignore_index=True,
)

rag_comparison_df

,model,question,reference_answer,rag_answer,correctness,groundedness,relevance,retrieval_relevance,retrieved_docs_count
0,gpt-4o-mini,How does the ReAct agent use self-reflection?,"ReAct integrates reasoning and acting, perform...",The ReAct agent uses self-reflection by integr...,True,True,True,True,6
1,gpt-4o-mini,What are the types of biases that can arise wi...,The biases that can arise with few-shot prompt...,The types of biases that can arise with few-sh...,True,True,True,True,6
2,gpt-4o-mini,What are five types of adversarial attacks?,Five types of adversarial attacks include toke...,The five types of adversarial attacks are toke...,True,True,True,True,6
3,gpt-4.1-mini,How does the ReAct agent use self-reflection?,"ReAct integrates reasoning and acting, perform...",The ReAct agent integrates reasoning and actin...,True,True,True,True,6
4,gpt-4.1-mini,What are the types of biases that can arise wi...,The biases that can arise with few-shot prompt...,The types of biases that can arise with few-sh...,True,True,True,True,6
5,gpt-4.1-mini,What are five types of adversarial attacks?,Five types of adversarial attacks include toke...,The five types of adversarial attacks are: 1) ...,True,True,True,True,6


In [17]:
rag_summary_df = rag_comparison_df.groupby("model").agg(
    total_examples=("question", "count"),
    correctness_score=("correctness", "mean"),
    groundedness_score=("groundedness", "mean"),
    relevance_score=("relevance", "mean"),
    retrieval_relevance_score=("retrieval_relevance", "mean"),
).reset_index()

rag_summary_df

,model,total_examples,correctness_score,groundedness_score,relevance_score,retrieval_relevance_score
0,gpt-4.1-mini,3,1.0,1.0,1.0,1.0
1,gpt-4o-mini,3,1.0,1.0,1.0,1.0
